### citations: 

Pham, H. H., Nguyen Trung, H., & Nguyen, H. Q. (2022). VinDr-Mammo: A large-scale benchmark dataset for computer-aided detection and diagnosis in full-field digital mammography (version 1.0.0). PhysioNet. https://doi.org/10.13026/br2v-7517.

Goldberger, A., Amaral, L., Glass, L., Hausdorff, J., Ivanov, P. C., Mark, R., ... & Stanley, H. E. (2000). PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals. Circulation [Online]. 101 (23), pp. e215–e220.

original dataset link: https://www.physionet.org/content/vindr-mammo/1.0.0/ 

Helpful references:

Visualization utilities: https://docs.pytorch.org/vision/main/auto_examples/others/plot_visualization_utils.html#

fasterrcnn_resnet50_fpn: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html

TV Tensors: https://docs.pytorch.org/vision/main/tv_tensors.html

Example with Penn-Fudan (uses MaskRCNN but same in principle): https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html 



In [ ]:
import torch, torchvision

In [ ]:
import os
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/engine.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/utils.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_utils.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_eval.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/transforms.py")

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(f"running on {device}")
print(torch.cuda.device_count())

# 1. Import and Explore the Data

In [ ]:
import pandas as pd
import os

#read the csv file
csv_path  = "/kaggle/input/vindr-breast-level-annotations-csv/vindr_detection_v1_folds.csv"
metadata = pd.read_csv(csv_path)
metadata['finding_birads'] = metadata['finding_birads'].fillna(metadata['breast_birads'])
#metadata.head

In [ ]:
from torchvision.transforms import v2

trans = v2.Compose([
    v2.RGB(),
    #v2.Resize(),
    v2.RandomHorizontalFlip(p=0.5),
    #v2.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.98, 1.02), shear=2),
    v2.ToDtype(torch.float32, scale=True)
])

In [ ]:
from torchvision.io import decode_image

def load_image(ind, metadata):
    row = metadata.iloc[ind]
    file_root = "/kaggle/input/vindr-mammogram-dataset-dicom-to-png/images_png"
    study_id = row['patient_id']
    image_id = row['image_id']
    path = os.path.join(file_root, study_id, image_id)
    return decode_image(path)

In [ ]:
from torchvision.utils import draw_bounding_boxes
import matplotlib.pyplot as plt
import PIL
from torchvision import tv_tensors

ind = 200
row = metadata.iloc[ind]
file_root = "/kaggle/input/vindr-mammogram-dataset-dicom-to-png/images_png"
study_id = row['patient_id']
image_id = row['image_id']
ex_file_path = os.path.join(file_root, study_id, image_id)

img = load_image(ind, metadata)  

print(img.shape)

roi_group = metadata.groupby(['patient_id', 'image_id']).get_group((study_id, image_id))
boxes = torch.tensor(
    roi_group[['resized_xmin', 'resized_ymin', 'resized_xmax', 'resized_ymax']].values,
    dtype=torch.float32
)

boxes = tv_tensors.BoundingBoxes(
    boxes,
    format="XYXY",
    canvas_size=img.shape[-2:]
)

boxed_img = draw_bounding_boxes(
    img,
    boxes,
    width=3,
    colors="red"
)

trans_img, trans_boxes = trans(img, boxes)
print(trans_img.shape)

trans_boxed_img = draw_bounding_boxes(
    trans_img,
    trans_boxes,
    width=3,
    colors="red"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10))

ax1.imshow(boxed_img.permute(1, 2, 0))  # CHW to HWC
ax1.set_title("Original Image")
ax1.axis('off')

ax2.imshow(trans_boxed_img.permute(1, 2, 0))  # CHW to HWC
ax2.set_title("Transformed Image")
ax2.axis('off')

Let's plot the distribution of BIRADS evaluations in the dataset. It's very imbalanced!

In [ ]:
counts = metadata["breast_birads"].value_counts().sort_values(ascending=False)
plt.bar(counts.index, counts.values)
plt.title("Birads evaluation distribution")
plt.xlabel("birads rating")
plt.ylabel("count")

The following is not that useful for binary classification. For future purposes, if we want to update the model to more specifically predict lesion type, simply add self.label_map = label_map to the dataset class and assign labels to corresponding boxes in get_item. However, this is kind of hard to do because the lesion types are also imbalanced; some more complicated modification to the loss function may be necessary. 

In [ ]:
metadata.head()

In [ ]:
import ast
from collections import Counter

# Convert string representation to actual lists
metadata['finding_categories'] = metadata['finding_categories'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) and x != '' else []
)

finding_categories = metadata["finding_categories"].sum()
counts = Counter(finding_categories)

label_map = {cat: idx+1 for idx, cat in enumerate(counts.keys())}

metadata["category_most_rare"] = metadata["finding_categories"].apply(lambda categories: min(categories, key = lambda cat: counts[cat]) if len(categories) > 0 else None)

In [ ]:
metadata['category_most_rare'].value_counts()

In [ ]:
metadata['finding_categories'].value_counts()

In [ ]:
metadata['category_most_rare'].value_counts()

# 2. Dataset and Dataloader

In [ ]:
#split by train and test.
train_df = metadata[metadata['split']=="training"].copy()
test_df = metadata[metadata['split']=="test"].copy()

In [ ]:
print(len(train_df.groupby(['patient_id', 'image_id'])))
print(len(test_df.groupby(['patient_id', 'image_id'])))

In [ ]:
counts = train_df["breast_birads"].value_counts().sort_values(ascending=False)
plt.bar(counts.index, counts.values)
plt.title("Birads evaluation distribution (train set)")
plt.xlabel("birads rating")
plt.ylabel("count")

In [ ]:
counts = test_df["breast_birads"].value_counts().sort_values(ascending=False)
plt.bar(counts.index, counts.values)
plt.title("Birads evaluation distribution (test set)")
plt.xlabel("birads rating")
plt.ylabel("count")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

train_df['birads_int'] = train_df['breast_birads'].str.extract('(\d+)').astype(int)
patient_meta = train_df.groupby('patient_id')['birads_int'].max().reset_index() #df of patient id

# train_df_og_copy = train_df.copy() 

train_ids, val_ids = train_test_split(
    patient_meta['patient_id'], 
    test_size=0.25, #60-20-20
    stratify=patient_meta['birads_int'],
    random_state=42
)

train_df_final = train_df[train_df['patient_id'].isin(train_ids)].reset_index(drop=True)
val_df = train_df[train_df['patient_id'].isin(val_ids)].reset_index(drop=True)

In [ ]:
print(len(train_df_final['patient_id'].value_counts()))
print(len(val_df['patient_id'].value_counts()))

In [ ]:
counts = train_df_final["breast_birads"].value_counts().sort_values(ascending=False)
plt.bar(counts.index, counts.values)
plt.title("Birads evaluation distribution (final train set)")
plt.xlabel("birads rating")
plt.ylabel("count")

In [ ]:
counts = val_df["breast_birads"].value_counts().sort_values(ascending=False)
plt.bar(counts.index, counts.values)
plt.title("Birads evaluation distribution (validation set)")
plt.xlabel("birads rating")
plt.ylabel("count")

To simplify the process, I used a binary classification system, whereby all images without boxes (BIRADS 1 and 2) are considered "negative" and all images with boxes (BIRADS 3-5) are considered "positive". 

In [ ]:
#weight computation for weightedrandomsampler.
img_groups = train_df_final.groupby(["patient_id", "image_id"])
num_neg = 0
for (patient_id, image_id), group in img_groups:
    if (group['breast_birads'].isin(['BI-RADS 1', 'BI-RADS 2'])).all():
        num_neg = num_neg + 1
neg_weight = 1/num_neg 
pos_weight = 1/(len(img_groups)-num_neg)
print(neg_weight)
print(pos_weight)
print(len(img_groups))
print(num_neg)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.tv_tensors import Image, BoundingBoxes
import torchvision.ops as ops
import torch
import os

class VindrDataset(Dataset):
    def __init__(self, df, root_dir, transforms=None):
        self.df = df
        self.root_dir = root_dir
        self.transforms = transforms
        
        # Group by composite key and filter BI-RADS 1 cases
        self.groups = []
        for (patient_id, image_id), group in df.groupby(['patient_id', 'image_id']):
            # Case 1: BI-RADS 1 and 2 (no ROIs expected)
            if (group['breast_birads'].isin(['BI-RADS 1', 'BI-RADS 2'])).all():
                self.groups.append(((patient_id, image_id), None))
            # Case 2: Has ROIs (BI-RADS 3-5)
            else:
                self.groups.append(((patient_id, image_id), group))
    
    def __len__(self):
        return len(self.groups)
        
    def __getitem__(self, idx):
        (patient_id, image_id), roi_group = self.groups[idx]
        img_path = os.path.join(self.root_dir, patient_id, image_id)
        image = Image(decode_image(img_path))
        
        # Get image dimensions for clipping
        h, w = image.shape[-2:]
       
        if roi_group is None:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
        else:
            # 1. Load initial boxes
            boxes = torch.tensor(
                roi_group[['resized_xmin', 'resized_ymin', 'resized_xmax', 'resized_ymax']].astype(float).values,
                dtype=torch.float32
            )
            
            # 2. Safety Filter: Remove "Degenerate" boxes (zero width or height)
            # A box is kept only if xmax > xmin AND ymax > ymin
            keep = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
            boxes = boxes[keep]
            
            # 3. Handle Edge Case: If no boxes survive the filter
            if len(boxes) == 0:
                boxes = torch.zeros((0, 4), dtype=torch.float32)
                labels = torch.zeros(0, dtype=torch.int64)
                iscrowd = torch.zeros(0, dtype=torch.int64)
                area = torch.zeros(0, dtype=torch.float32)
            else:
                # 4. Safety Filter: Clip boxes to image boundaries [0, w] and [0, h]
                # Prevents RetinaNet from crashing on out-of-bounds coordinates
                boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, w)
                boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, h)
                
                # 5. Populate standard target fields
                labels = torch.ones(len(boxes), dtype=torch.int64) 
                iscrowd = torch.zeros(len(boxes), dtype=torch.int64)
                area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        
        # Wrap in TV Tensor for v2 transforms
        boxes = BoundingBoxes(
            boxes,
            format="XYXY",
            canvas_size=(h, w)
        )
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': int(idx),
            'area': area,
            'iscrowd': iscrowd
        }

        # Apply transforms (e.g., Flip, Affine) - automatically updates boxes
        if self.transforms:
            image, target = self.transforms(image, target)
            
        return image, target

In [ ]:
eval_trans = v2.Compose([
    v2.RGB(),
    v2.ToDtype(torch.float32, scale=True)
])

In [ ]:
train_set = VindrDataset(
    df = train_df_final,
    root_dir = file_root,
    transforms = trans
)
val_set = VindrDataset(
    df = val_df,
    root_dir = file_root,
    transforms = eval_trans
)

test_set = VindrDataset(
    df = test_df,
    root_dir = file_root,
    transforms = eval_trans
)

check image sample

In [ ]:
weights = [neg_weight if roi_group is None else pos_weight for _, roi_group in train_set.groups]
train_weights = torch.DoubleTensor(weights)

In [ ]:
from torch.utils.data import WeightedRandomSampler

TRAIN_BATCH_SIZE = 4
VAL_BATCH_SIZE = 8
TEST_BATCH_SIZE = 8

train_loader = DataLoader(
    dataset = train_set,
    batch_size = TRAIN_BATCH_SIZE,
    sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights)),
    collate_fn=lambda batch: tuple(zip(*batch)),
    num_workers = 4,
    pin_memory = True,
)

val_loader = DataLoader(
    dataset = val_set,
    batch_size = VAL_BATCH_SIZE,
    collate_fn=lambda batch: tuple(zip(*batch)),
    shuffle = False
)

test_loader = DataLoader(
    dataset = test_set,
    batch_size = TEST_BATCH_SIZE,
    collate_fn=lambda batch: tuple(zip(*batch)),
    shuffle = False
)

# 3. Model, Optimizer, and Scheduler

**Things to try:**

1. Benchmark different object detection models (Faster-RCNN, Retinanet, YOLO).
2. training from scratch (with 0.5-normalized or normalized on dataset stats) vs pretrained COCO weights (with Imagenet normalization)
3. undersampling the negatives vs no undersampling.
4. Binary classification vs classification by lesion type.
5. Experiment with learning rate and scheduler. 

**Currently trying:** 
Retinanet, COCO weights w default Imagenet normalization, no undersampling, binary classification.

Engine.py script is from Torchvision references

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR
import torch.nn as nn

model = retinanet_resnet50_fpn_v2(weights="DEFAULT")
custom_transform = torchvision.models.detection.transform.GeneralizedRCNNTransform(
    min_size=480, 
    max_size=640,
    image_mean=[0.485, 0.456, 0.406],
    image_std = [0.229, 0.224, 0.225]
)

model.transform = custom_transform

num_classes = 2 #lesion + background
num_anchors = model.head.classification_head.num_anchors
model.head.classification_head = RetinaNetClassificationHead(
    in_channels=256, 
    num_anchors=num_anchors, 
    num_classes=num_classes
)

model.to(device)

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

optimizer = SGD(
    params=model.parameters(), 
    lr=0.0001, momentum=0.9, 
    weight_decay=1e-3
)

lr_scheduler = StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

print(model)

# 4. Training and Evaluation Loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0, path='best_model.pth'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.best_map = None
        self.no_improvement_count = 0
        self.stop_training = False

    def check_early_stop(self, cur_map, model):
        # We want mAP to increase, so we check if current is better than best + delta
        if self.best_map is None or cur_map > (self.best_map + self.delta):
            self.best_map = cur_map
            self.no_improvement_count = 0  # Fixed: added self.
            self.save_checkpoint(model)
        else:
            self.no_improvement_count += 1
            print(f"EarlyStopping counter: {self.no_improvement_count} out of {self.patience}")
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                print("[-------STOPPING TRAINING: PATIENCE REACHED-------]")

    def save_checkpoint(self, model):
        # Use the path defined in __init__
        torch.save(model.state_dict(), self.path)
        print(f"Validation mAP improved. Saved model to {self.path}")

In [ ]:
import engine 
import gc

NUM_EPOCHS = 25
scaler = torch.amp.GradScaler('cuda')
all_avg_losses = []
all_scores = []

early_stopper = EarlyStopping(patience=5, delta=0.001)

print("----------training starts----------")
for epoch in range(NUM_EPOCHS):
    print(f"\n----------------training for epoch {epoch}----------------")
    
    metric_logger = engine.train_one_epoch(model, optimizer, train_loader, device, epoch, print_freq=100, scaler=scaler)
    all_avg_losses.append(metric_logger.loss.global_avg)
    lr_scheduler.step()

    if 'eval' in locals(): del eval 
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"---------------evaluation for epoch {epoch}---------------")
    eval = engine.evaluate(model, val_loader, device=device)
    cur_map = eval.coco_eval['bbox'].stats[0]
    early_stopper.check_early_stop(cur_map, model)
    all_scores.append(cur_map)

    if early_stopper.stop_training:
        break

print("\n--------------done!----------------")

print("Loading best model for final evaluation...")
model.load_state_dict(torch.load('best_model.pth'))

In [ ]:
%tb

In [ ]:
engine.evaluate(model, test_loader, device=device)

# 5. Future Considerations

With a functional ROI detection model, we may also seek to classify the lesion type or possibly BIRADS severity. A future pipeline could include using the ROI prediction model to crop mammogram images, then feed into a separate whole-image classifier model. 